In [1]:
import cv2
import tensorflow as tf
import numpy as np
import os

# Load the trained model
model_path = r"C:\essat\projet ia\detectation de maladie\trained_model.h5"
if not os.path.exists(model_path):
    print(f"Error: Model file not found at {model_path}")
    exit()
model = tf.keras.models.load_model(model_path)

# Define a function to preprocess frames for the model
def preprocess_frame(frame):
    input_size = (128, 128)  # Adjust based on your model's input size
    resized_frame = cv2.resize(frame, input_size)
    normalized_frame = resized_frame / 255.0  # Normalize the frame
    return np.expand_dims(normalized_frame, axis=0)  # Add batch dimension

# Start video capture
cap = cv2.VideoCapture(0)
if not cap.isOpened():
    print("Error: Could not open camera.")
    exit()

# Dynamically generate class labels from the dataset directory
dataset_path = r"C:\essat\projet ia\detectation de maladie\dataset\New Plant Diseases Dataset(Augmented)\New Plant Diseases Dataset(Augmented)\train"
if not os.path.exists(dataset_path):
    print(f"Error: Dataset path not found at {dataset_path}")
    exit()
class_labels = sorted(os.listdir(dataset_path))

while True:
    ret, frame = cap.read()
    if not ret:
        print("Error: Failed to capture frame.")
        break

    # ---------- DETECTION DE LA FEUILLE PAR CONTOUR ----------
    # Convert to grayscale
    gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)

    # Apply Gaussian blur
    blurred = cv2.GaussianBlur(gray, (5, 5), 0)

    # Edge detection
    edges = cv2.Canny(blurred, 50, 150)

    # Find contours
    contours, _ = cv2.findContours(edges, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)

    # Filtrer les contours avec surface minimale
    leaf_detected = False
    for cnt in contours:
        area = cv2.contourArea(cnt)
        if area > 1000:  # Seuil pour considérer une feuille
            x, y, w, h = cv2.boundingRect(cnt)
            cv2.rectangle(frame, (x, y), (x + w, y + h), (255, 0, 0), 2)
            leaf_detected = True

    if not leaf_detected:
        # Aucun contour significatif → pas de feuille
        cv2.putText(frame, "NO LEAF", (10, 60),
                    cv2.FONT_HERSHEY_SIMPLEX, 1, (0, 0, 255), 2)
    else:
        # ---------- PREPROCESSING POUR LE MODELE ----------
        input_data = preprocess_frame(frame)

        # ---------- PREDICTION ----------
        try:
            predictions = model.predict(input_data)
        except Exception as e:
            print(f"Error during prediction: {e}")
            break

        predicted_index = np.argmax(predictions)
        if predicted_index >= len(class_labels):
            print(f"Error: Predicted index {predicted_index} out of range.")
            break

        predicted_disease = class_labels[predicted_index]

        # Affichage du résultat
        cv2.putText(frame, f"Disease: {predicted_disease}", (10, 30),
                    cv2.FONT_HERSHEY_SIMPLEX, 1, (0, 255, 0), 2)

    # Affichage du frame
    cv2.imshow("Real-Time Disease Detection", frame)

    # Quitter avec la touche 'q'
    if cv2.waitKey(1) & 0xFF == ord('q'):
        break

# Libération des ressources
cap.release()
cv2.destroyAllWindows()


c:\Users\mohamed\anaconda3\Lib\site-packages\keras\src\layers\convolutional\base_conv.py:107: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)
c:\Users\mohamed\anaconda3\Lib\site-packages\keras\src\optimizers\base_optimizer.py:86: UserWarning: Argument `decay` is no longer supported and will be ignored.
  warnings.warn(


1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 432ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 65ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 68ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 64ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 75ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 66ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 60ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 60ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 62ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 80ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 64ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 54ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 59ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 70ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 67ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 75ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 52ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 59ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 62ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 56ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 58ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 63ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 61ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 61ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 58ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 63ms/step
1/1 ━━━━━━━